## Evaluation, Critical Analysis and Ethics

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

In [ ]:
# Load clustered dataset
df = pd.read_csv(
    '/Users/meecee/Desktop/Github/Networking Recommendation System/data/processed/clustered_member_profiles.csv')

In [ ]:
# Cold-start analysis: how many members have 0 recorded events or unknown titles/sectors?
zero_events = (df['event_attendance_count'] == 0).sum()
unknown_titles = (df['seniority_level_name'] == 'Unknown').sum()
unknown_sectors = (df['clean_sector'] == 'Unknown').sum()

print("Cold-Start & Sparsity Audit:")
print(f"Total members: {len(df)}")
print(
    f"Members with 0 recorded events: {zero_events} ({zero_events/len(df)*100:.1f}%)")
print(
    f"Members with unknown job titles: {unknown_titles} ({unknown_titles/len(df)*100:.1f}%)")
print(
    f"Members with unknown sectors: {unknown_sectors} ({unknown_sectors/len(df)*100:.1f}%)")

### Exposure & Superstar Bias check

In [ ]:
# Let's see how many times top organizations or senior members appear if we run recommendations across different cohorts
org_counts = df['organization'].dropna().value_counts().head(10)
print("\nTop 10 Represented Organizations:")
display(org_counts.to_frame().rename(columns={'organization': 'Member Count'}))

### Data Analysis: Temporal and Thematic Trends

In [ ]:
# Let's inspect attendance_df and event names to discover trends
att_df = pd.read_csv(
    '/Users/meecee/Desktop/Github/Networking Recommendation System/data/processed/event_attendance.csv')
print("Total attendance entries:", len(att_df))

In [ ]:
# Breakdown of event attendance by thematic keywords
keywords = {
    'AI & Data': ['ai', 'artificial intelligence', 'dystopia', 'utopia', 'data', 'future10'],
    'Cyber Defence & Resilience': ['cyber', 'resili', 'breaking point', 'defence', 'guardians', 'space'],
    'Ecosystem & Innovation': ['ecosystem', 'collaboration', 'partnership', 'innovation', 'emerging', 'pic n mix'],
    'Startups & Growth': ['startup', 'growth', 'scale', 'showcase']
}


def tag_event(title):
    t = str(title).lower()
    tags = []
    for category, terms in keywords.items():
        if any(term in t for term in terms):
            tags.append(category)
    return tags if tags else ['General Networking / Regional']


att_df['tags'] = att_df['event_name'].apply(tag_event)
# Explode tags
exploded_tags = att_df.explode('tags')
print("\nEvent Category Attendance Breakdown:")
cat_counts = exploded_tags['tags'].value_counts()
display(cat_counts.to_frame())

In [ ]:
# Let's see cross-sector event attendance
merged_att = att_df.merge(
    df[['email', 'clean_sector', 'seniority_level_name', 'cluster']], on='email', how='left')
print("\nAttendance by Sector:")
display(merged_att['clean_sector'].value_counts().to_frame())

In [ ]:
# Cross-tabulation between Sector and Theme
theme_by_sec = pd.crosstab(merged_att['clean_sector'], merged_att['event_name'].apply(
    lambda x: tag_event(x)[0]), normalize='index') * 100
print("\nTheme distribution per sector (%):")
display(theme_by_sec.round(1))

In [ ]:
# Plotting Strategic Insight Visualizations
plt.figure(figsize=(10, 5))
cat_counts_clean = cat_counts.drop('General Networking / Regional')
sns.barplot(x=cat_counts_clean.values, y=cat_counts_clean.index,
            hue=cat_counts_clean.index, palette='viridis', legend=False)
plt.title('Thematic Engagement Beyond General Networking (CyNam Ecosystem)',
          fontsize=13, fontweight='bold')
plt.xlabel('Total Event Attendance Records')
plt.ylabel('Thematic Focus Area')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('/Users/meecee/Desktop/Github/Networking Recommendation System/assets/thematic_engagement_trends.png', dpi=300)
plt.show()
plt.close()

In [ ]:
# Plotting Organization Presence
plt.figure(figsize=(10, 5))
sns.barplot(x=org_counts.values, y=org_counts.index,
            hue=org_counts.index, palette='crest', legend=False)
plt.title('Top 10 Organizations by Member Volume in CyNam Ecosystem',
          fontsize=13, fontweight='bold')
plt.xlabel('Registered Members Count')
plt.ylabel('Organization')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('/Users/meecee/Desktop/Github/Networking Recommendation System/assets/top_organizations_distribution.png', dpi=300)
plt.show()
plt.close()
print("Saved strategic insight charts.")